<a href="https://colab.research.google.com/github/lucasmvaladao/dimensionamento-mercado-varejo-farmaceutico/blob/main/01_pipeline_etl_dimensionamento_mercado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto 1: Inteligência de Mercado e Expansão Territorial no Varejo Farmacêutico (SP)

**Objetivo:** Construir um pipeline analítico de ponta a ponta para dimensionar o mercado potencial (TAM, SAM e SOM) e mapear oportunidades de expansão física no Estado de São Paulo através de modelagem dimensional (Kimball).

**Arquitetura do Pipeline:**
0. Preparação de dependências e autenticação de segurança em nuvem.
1. Configuração de bibliotecas e ambiente de execução.
2. Ingestão de dados demográficos e hierarquia regional (APIs do IBGE).
3. Extração de estabelecimentos concorrentes da RAIS (BigQuery / Base dos Dados).
4. Engenharia de métricas de mercado (TAM, SAM, SOM e densidade concorrencial).
5. Estruturação dimensional (1 Fato + 5 Dimensões) e carga automatizada no Google BigQuery.

## 0. Instalação de Dependências e Autenticação no GCP
Instalação dos pacotes de extração externa (`sidrapy`, `basedosdados`) e emissão do token OAuth2 para interação com as APIs do Google Cloud Platform.

In [28]:
# Instalação silenciosa de pacotes essenciais
!pip install pandas sidrapy basedosdados -q

# Autenticação de credenciais da conta Google no Colab
from google.colab import auth
auth.authenticate_user(clear_output=False)

print("Dependências instaladas e usuário autenticado com sucesso no Google Cloud.")

Dependências instaladas e usuário autenticado com sucesso no Google Cloud.


## 1. Configuração de Ambiente e Inicialização do Cliente BigQuery
Importação dos módulos de análise e conexão do cliente ao projeto `projeto-mercado-portfolio`.

In [29]:
import basedosdados as bd
from google.cloud import bigquery
import numpy as np
import pandas as pd
import requests

PROJECT_ID = "projeto-mercado-portfolio"
DATASET_ID = "mercado_sp"

client = bigquery.Client(project=PROJECT_ID)
print("Cliente BigQuery instanciado e pronto para operação.")

Cliente BigQuery instanciado e pronto para operação.


## 2. Ingestão Demográfica e Hierarquia Regional (IBGE)
Coleta de dados primários governamentais via APIs REST do IBGE:
* **População:** Resultados consolidados do Censo Demográfico 2022 (Tabela 9514).
* **Estrutura Territorial:** Mapeamento de microrregiões e mesorregiões oficiais do Estado de São Paulo.

In [30]:
print("Coletando População Municipal (Censo 2022)...")
url_pop = "https://servicodados.ibge.gov.br/api/v3/agregados/9514/periodos/2022/variaveis/93?localidades=N6[N3[35]]"
res_pop = requests.get(url_pop).json()
df_pop = pd.DataFrame([
    {
        "id_municipio_ibge": str(item["localidade"]["id"]),
        "nome_municipio": item["localidade"]["nome"],
        "populacao": int(list(item["serie"].values())[0]),
    }
    for item in res_pop[0]["resultados"][0]["series"]
])

print("Coletando Hierarquia Regional (Meso e Microrregiões)...")
url_reg = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/35/municipios"
res_reg = requests.get(url_reg).json()
df_regioes = pd.DataFrame([
    {
        "id_municipio_ibge": str(item["id"]),
        "id_microrregiao": str(item["microrregiao"]["id"]),
        "nome_microrregiao": item["microrregiao"]["nome"],
        "id_mesorregiao": str(item["microrregiao"]["mesorregiao"]["id"]),
        "nome_mesorregiao": item["microrregiao"]["mesorregiao"]["nome"],
    }
    for item in res_reg
])

print(f"Demografia e malha regional carregadas: {len(df_pop)} municípios.")

Coletando População Municipal (Censo 2022)...
Coletando Hierarquia Regional (Meso e Microrregiões)...
Demografia e malha regional carregadas: 645 municípios.


## 3. Extração Concorrencial da RAIS (Google BigQuery)
Consulta SQL aos microdados de estabelecimentos ativos no Estado de São Paulo, filtrando pelo CNAE `4771701` (Comércio varejista de produtos farmacêuticos).

In [31]:
print("Consultando microdados da RAIS no BigQuery via Base dos Dados...")
query_rais = """
SELECT
    id_municipio,
    cnae_2,
    COUNT(*) AS total_estabelecimentos
FROM `basedosdados.br_me_rais.microdados_estabelecimentos`
WHERE ano = 2021
  AND sigla_uf = 'SP'
  AND cnae_2 = '4771701'
GROUP BY id_municipio, cnae_2
"""
df_rais = bd.read_sql(query_rais, billing_project_id=PROJECT_ID, reauth=False)
df_rais["id_municipio"] = df_rais["id_municipio"].astype(str).str.strip()
df_rais["total_estabelecimentos"] = pd.to_numeric(
    df_rais["total_estabelecimentos"], errors="coerce"
).fillna(0)

print(f"RAIS carregada: {len(df_rais)} municípios com estabelecimentos ativos no setor.")

Consultando microdados da RAIS no BigQuery via Base dos Dados...
Downloading: |          |
RAIS carregada: 0 municípios com estabelecimentos ativos no setor.


## 4. Engenharia de Variáveis e Dimensionamento de Mercado
Cruzamento relacional das bases para calcular indicadores estratégicos orientados a P&L:
* **Densidade Concorrencial:** Lojas ativas por 1.000 habitantes.
* **TAM (Total Addressable Market):** Gasto potencial anual municipal (R$ 850,00 per capita).
* **SAM (Serviceable Available Market):** Demanda restrita a praças com viabilidade de rede ($\ge$ 50 mil hab).
* **SOM (Serviceable Obtainable Market):** Meta conservadora de captura (4% do SAM).

In [32]:
print("Cruzando bases e calculando indicadores de mercado...")
df_full = df_pop.merge(df_regioes, on="id_municipio_ibge", how="left")
df_full = df_full.merge(df_rais, left_on="id_municipio_ibge", right_on="id_municipio", how="left")

df_full["total_estabelecimentos"] = df_full["total_estabelecimentos"].fillna(0).astype(int)
df_full["cnae_2"] = df_full["cnae_2"].fillna("4771701")

# Densidade de concorrência
df_full["densidade_lojas_mil_hab"] = np.where(
    df_full["populacao"] > 0,
    (df_full["total_estabelecimentos"] / df_full["populacao"]) * 1000,
    0,
)

# Métricas financeiras
df_full["tam_reais"] = df_full["populacao"] * 850.0
df_full["sam_reais"] = np.where(df_full["populacao"] >= 50000, df_full["tam_reais"], 0.0)
df_full["som_reais"] = df_full["sam_reais"] * 0.04

print("Engenharia de variáveis finalizada com sucesso.")

Cruzando bases e calculando indicadores de mercado...
Engenharia de variáveis finalizada com sucesso.


## 5. Modelagem Dimensional (Kimball)
Separação das entidades analíticas em um esquema estrela canônico composto por 5 Dimensões e 1 Fato:
* `dim_municipio`: Cadastro de cidades e chave regional.
* `dim_regiao`: Mesorregiões, microrregiões e agrupamento em macroclusters.
* `dim_cnae`: Atividades econômicas e categorização de varejo.
* `dim_tempo`: Metadados temporais e safras das fontes.
* `dim_perfil_socioeconomico`: Estratificação de porte populacional e potencial de consumo.
* `fato_densidade_varejo`: Métricas aditivas consolidadas e chaves estrangeiras (FKs).

In [33]:
print("Construindo tabelas dimensionais...")

# Dimensão Município
dim_municipio = df_full[["id_municipio_ibge", "nome_municipio", "id_microrregiao"]].drop_duplicates()

# Dimensão Região
dim_regiao = df_regioes[["id_microrregiao", "nome_microrregiao", "id_mesorregiao", "nome_mesorregiao"]].drop_duplicates()
dim_regiao["macro_cluster"] = np.where(
    dim_regiao["nome_mesorregiao"].str.contains("Metropolitana", case=False, na=False),
    "Região Metropolitana de SP",
    "Interior & Litoral Paulista",
)

# Dimensão CNAE
dim_cnae = pd.DataFrame([
    {
        "cnae_2": "4771701",
        "nome_cnae": "Comércio varejista de produtos farmacêuticos, sem manipulação de fórmulas",
        "divisao_cnae": "Comércio Varejista",
        "setor_estrategico": "Farmácias & Saúde",
    },
    {
        "cnae_2": "4789004",
        "nome_cnae": "Comércio varejista de animais vivos e de alimentos para animais",
        "divisao_cnae": "Comércio Varejista",
        "setor_estrategico": "Pet Shops & Cuidados",
    },
])

# Dimensão Tempo
dim_tempo = pd.DataFrame([{
    "ano": 2021,
    "decada": "2020s",
    "safra_estabelecimentos": "RAIS 2021",
    "safra_demografia": "Censo Demográfico 2022",
}])

# Dimensão Perfil Socioeconômico
dim_perfil_socioeconomico = pd.DataFrame([
    {"id_perfil": 1, "porte_populacional": "Micro (Até 20k hab)", "potencial_consumo": "Consumo Local Reduzido"},
    {"id_perfil": 2, "porte_populacional": "Pequeno (20k a 50k hab)", "potencial_consumo": "Consumo Emergente"},
    {"id_perfil": 3, "porte_populacional": "Médio (50k a 100k hab)", "potencial_consumo": "Polo Microrregional"},
    {"id_perfil": 4, "porte_populacional": "Grande (100k a 500k hab)", "potencial_consumo": "Alta Atratividade Comercial"},
    {"id_perfil": 5, "porte_populacional": "Metrópole (Acima de 500k hab)", "potencial_consumo": "Hiperconcentrado / Alta Saturação"},
])

# Mapear chave estrangeira id_perfil na Fato
condicoes = [
    df_full["populacao"] < 20000,
    df_full["populacao"] < 50000,
    df_full["populacao"] < 100000,
    df_full["populacao"] < 500000,
]
escolhas = [1, 2, 3, 4]
df_full["id_perfil"] = np.select(condicoes, escolhas, default=5)

# Tabela Fato
fato_densidade_varejo = pd.DataFrame({
    "id_municipio_ibge": df_full["id_municipio_ibge"],
    "cnae_2": df_full["cnae_2"],
    "ano": 2021,
    "id_perfil": df_full["id_perfil"].astype(int),
    "populacao": df_full["populacao"].astype(int),
    "total_estabelecimentos": df_full["total_estabelecimentos"].astype(int),
    "densidade_lojas_mil_hab": df_full["densidade_lojas_mil_hab"].round(4),
    "tam_reais": df_full["tam_reais"].round(2),
    "sam_reais": df_full["sam_reais"].round(2),
    "som_reais": df_full["som_reais"].round(2),
})

print("Esquema estrela modelado com sucesso.")

Construindo tabelas dimensionais...
Esquema estrela modelado com sucesso.


## 6. Carga Automatizada no Google BigQuery
Sobrescrita controlada (`WRITE_TRUNCATE`) das tabelas dimensionais no dataset de produção `mercado_sp` para conexão direta com o Power BI Desktop.

In [34]:
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
client.create_dataset(bigquery.Dataset(dataset_ref), exists_ok=True)
job_config = bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)

tabelas = {
    "fato_densidade_varejo": fato_densidade_varejo,
    "dim_municipio": dim_municipio,
    "dim_regiao": dim_regiao,
    "dim_cnae": dim_cnae,
    "dim_tempo": dim_tempo,
    "dim_perfil_socioeconomico": dim_perfil_socioeconomico,
}

print(f"Enviando tabelas para o BigQuery ({DATASET_ID})...")
for nome, df in tabelas.items():
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{nome}"
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    print(f"-> Tabela '{nome}' carregada com sucesso ({len(df)} registros)!")

print("\nPipeline finalizado! O esquema estrela está pronto no BigQuery para ingestão no Power BI.")

Enviando tabelas para o BigQuery (mercado_sp)...
-> Tabela 'fato_densidade_varejo' carregada com sucesso (645 registros)!
-> Tabela 'dim_municipio' carregada com sucesso (645 registros)!
-> Tabela 'dim_regiao' carregada com sucesso (63 registros)!
-> Tabela 'dim_cnae' carregada com sucesso (2 registros)!
-> Tabela 'dim_tempo' carregada com sucesso (1 registros)!
-> Tabela 'dim_perfil_socioeconomico' carregada com sucesso (5 registros)!

Pipeline finalizado! O esquema estrela está pronto no BigQuery para ingestão no Power BI.
